# TaxPro-CL taxonomy rank-impact -- quick look

Loads `result/ranking_summary.csv` and `result/ranking_details.csv` (produced by `run_comparison.py`).
No pandas dependency -- this project's requirements.txt does not include it, so everything here is stdlib `csv` + `matplotlib`.

In [ ]:
import csv
from pathlib import Path

RESULT_DIR = Path("result")
SUMMARY_CSV = RESULT_DIR / "ranking_summary.csv"
DETAILS_CSV = RESULT_DIR / "ranking_details.csv"

NUMERIC_SUMMARY_FIELDS = {
    "K", "Increased", "Decreased", "Unchanged", "Enter_TopK", "Leave_TopK",
    "Increased_Ratio", "Avg_Delta_Rank", "Median_Delta_Rank", "Std_Delta_Rank",
    "N_Items", "N_Users", "Baseline_Validation_Score", "TaxProCL_Validation_Score",
}

def load_rows(path, numeric_fields=()):
    rows = []
    with path.open("r", encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            for field in numeric_fields:
                if field in row and row[field] != "":
                    row[field] = float(row[field])
            rows.append(row)
    return rows

summary = load_rows(SUMMARY_CSV, NUMERIC_SUMMARY_FIELDS)
print(len(summary), "summary rows")

In [ ]:
# Dataset x Baseline x K -> Increased_Ratio / Avg_Delta_Rank, sorted for scanning.
header = f"{'Dataset':<28}{'Baseline':<10}{'K':>4}  {'Increased_Ratio':>16}  {'Avg_Delta_Rank':>15}  {'N_Items':>10}"
print(header)
print("-" * len(header))
for row in sorted(summary, key=lambda r: (r["Dataset"], r["Baseline"], r["K"])):
    print(
        f"{row['Dataset']:<28}{row['Baseline']:<10}{int(row['K']):>4}  "
        f"{row['Increased_Ratio']:>16.3f}  {row['Avg_Delta_Rank']:>15.2f}  {int(row['N_Items']):>10}"
    )

In [ ]:
import matplotlib.pyplot as plt

# Pick one (Dataset, Baseline, K) combo to inspect in detail -- edit these to explore others.
TARGET_DATASET = summary[0]["Dataset"] if summary else None
TARGET_BASELINE = summary[0]["Baseline"] if summary else None
TARGET_K = int(summary[0]["K"]) if summary else None
print("Inspecting:", TARGET_DATASET, TARGET_BASELINE, "K=", TARGET_K)

deltas = []
if TARGET_DATASET is not None:
    with DETAILS_CSV.open("r", encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            if (
                row["Dataset"] == TARGET_DATASET
                and row["Baseline"] == TARGET_BASELINE
                and int(row["K"]) == TARGET_K
            ):
                deltas.append(int(row["Delta_Rank"]))

print(len(deltas), "rows loaded")
if deltas:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(deltas, bins=50)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Delta_Rank (Rank_Baseline - Rank_TaxProCL, positive = TaxPro-CL ranks it better)")
    ax.set_ylabel("count")
    ax.set_title(f"{TARGET_DATASET} / {TARGET_BASELINE} / K={TARGET_K}")
    plt.show()